# RAG With Hatstack 2.0

**Author:** Shinin Varongchayakul

**Date:** 21 Jun 2026

Dataset from: https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive

## 1 Prepare Documents

### 1.1 Load CSV

In [32]:
# Import package
import pandas as pd

# Load CSV
articles_df = pd.read_csv("bbc-news-data.csv", sep="\t")

### 1.2 Explore Dataset

In [33]:
# Preview df
articles_df.head

<bound method NDFrame.head of       category filename                              title  \
0     business  001.txt  Ad sales boost Time Warner profit   
1     business  002.txt   Dollar gains on Greenspan speech   
2     business  003.txt  Yukos unit buyer faces loan claim   
3     business  004.txt  High fuel prices hit BA's profits   
4     business  005.txt  Pernod takeover talk lifts Domecq   
...        ...      ...                                ...   
2220      tech  397.txt   BT program to beat dialler scams   
2221      tech  398.txt    Spam e-mails tempt net shoppers   
2222      tech  399.txt            Be careful how you code   
2223      tech  400.txt    US cyber security chief resigns   
2224      tech  401.txt   Losing yourself in online gaming   

                                                content  
0      Quarterly profits at US media giant TimeWarne...  
1      The dollar has hit its highest level against ...  
2      The owners of embattled Russian oil giant Yu

In [34]:
# Get df info
articles_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   category  2225 non-null   str  
 1   filename  2225 non-null   str  
 2   title     2225 non-null   str  
 3   content   2225 non-null   str  
dtypes: str(4)
memory usage: 69.7 KB


In [35]:
# View one article
print(articles_df.loc[0, "title"])
print()
print(articles_df.loc[0, "content"][:1_000])

Ad sales boost Time Warner profit

 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AO

### 1.3 Covert to Document

In [36]:
# Import package
from haystack import Document

# Create a Document collector
documents = []

# Create Documents
for row in articles_df.itertuples():
    doc = Document(
        content=row.content,
        meta={
            "title": row.title,
            "category": row.category,
            "date": row.filename
        }
    )
    documents.append(doc)

In [37]:
# Inspect first Document
documents[0]

Document(id=3f49cfe7b451f9d387da3f2fc9c6bd1f12155d3d83adb7f4a94498af9ed3f193, content: ' Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months t...', meta: {'title': 'Ad sales boost Time Warner profit', 'category': 'business', 'date': '001.txt'})

## 2. Split Documents

In [38]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/shivarong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [39]:
# Import package
from haystack.components.preprocessors import RecursiveDocumentSplitter

# Instantiate splitter
splitter = RecursiveDocumentSplitter(
    split_length=180,
    split_overlap=30,
    split_unit="word",
    separators=[
        "\n\n",
        "sentence",
        "\n",
        " "
    ]
)

# Split Documents
splitted_documents = splitter.run(documents=documents)

# Get chunks
chunks = splitted_documents["documents"]

In [40]:
# Inspect first chunk
print(chunks[0].content)

print("\n--- Metadata ---")
print(chunks[0].meta)

 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for high-spe

## 3. Embed Chunks

### 3.1 Create Embedder

In [48]:
import sys

!{sys.executable} -m pip install -U sentence-transformers-haystack

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 930.7 kB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 1.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 1.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 1.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 1.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 1.3 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 1.4 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 1.6 MB/s  0:00:58m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 1.8 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 3.0 MB/s  0:00:0036m-:--:--
  Attempting uninstall: tokenizers╸━━━━━━━━━━━━━━  9/14 [scikit-learn]
    Found existing installation: tokenizers 0.23.1━━━━━━━━━━

In [49]:
# Import packages
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder
)

# Create embedder
document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2",
    meta_fields_to_embed=["title"],
)

### 3.2 Embed Chunks

In [52]:
# Embed chunks
embedding_result = document_embedder.run(documents=chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/225 [00:00<?, ?it/s]

In [53]:
# Get embedded chunks
embedded_chunks = embedding_result["documents"]

In [54]:
# Inspect one embedding
print(
    f"Embedding length: "
    f"{len(embedded_chunks[0].embedding):,}"
)

Embedding length: 384


## 4. Store Embeddings

### 4.1 Create Vector Store

In [56]:
# Import packages
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore

# Create in-memory vector store
document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")

### 4.2 Write Documents to Store

In [57]:
# Create Document Writer
document_writer = DocumentWriter(document_store=document_store)

In [58]:
# Write documents to store
write_result = document_writer.run(documents=embedded_chunks)

In [59]:
# Write the number of chunks written and stored
print(
    f"Chunks written: "
    f"{write_result['documents_written']:,}"
)

print(
    f"Chunks currently stored: "
    f"{document_store.count_documents():,}"
)

Chunks written: 7,182
Chunks currently stored: 7,182


## 5. Create Retriever

### 5.1 Create

In [60]:
# Create embedder for question
question_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

In [64]:
# Import package
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

# Create retriever
retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=5
)

### 5.2 Test

In [70]:
# Set test question
question = "What concerns did users raise about Google’s AutoLink feature?"

# Embed question
question_embedding_result = question_embedder.run(text=question)

# Get question embedding
embedded_question = question_embedding_result["embedding"]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [71]:
# Retrieve relevant documents
retrieval_result = retriever.run(query_embedding=embedded_question)

# Get documents
retrieved_documents = retrieval_result["documents"]

In [73]:
# Display retrieved documents
for number, document in enumerate(retrieved_documents, start=1):
    print("=" * 50)
    print(f"SOURCE {number}")
    print(f"Title: {document.meta.get('title', 'Unknown title')}")
    print(f"Category: {document.meta.get('category', 'Unknown category')}")
    print(f"File: {document.meta.get('filename', 'Not available')}")
    print()
    print(document.content[:700])
    print()

SOURCE 1
Title: Google's toolbar sparks concern
Category: tech
File: Not available

 Search engine firm Google has released a trial tool which is concerning some net users because it directs people to pre-selected commercial websites.  The AutoLink feature comes with Google's latest toolbar and provides links in a webpage to Amazon.com if it finds a book's ISBN number on the site. It also links to Google's map service, if there is an address, or to car firm Carfax, if there is a licence plate. Google said the feature, available only in the US, "adds useful links". But some users are concerned that Google's dominant position in the search engine market place could mean it would be giving a competitive edge to firms like Amazon.  AutoLink works by creating a link to a website

SOURCE 2
Title: Google's toolbar sparks concern
Category: tech
File: Not available

that is looking to continue its hypergrowth". In a statement Google said the feature was still only in beta, ie trial, stage and t

## 6. Generate Response

### 6.1 Build Prompt Template

In [76]:
# Import packages
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.generators.google_genai import GoogleGenAIChatGenerator

# Create system prompt
system_prompt = """
You are a helpful assistant for a historical BBC News Archive.

Answer only from the retrieved BBC article passages.

Rules:
- Do not use outside knowledge.
- Do not invent facts that are not supported by the passages.
- The archive is historical, so do not present its content as current news.
- Cite factual claims using source labels such as [1] or [2].
- If the retrieved passages do not contain enough evidence,
  say that clearly.
"""

# Create user prompt
user_prompt = """
Question:
{{ question }}

Retrieved BBC archive passages:

{% for doc in documents %}
[{{ loop.index }}]

Title: {{ doc.meta["title"] }}
Category: {{ doc.meta["category"] }}
Filename: {{ doc.meta["filename"] }}

Passage:
{{ doc.content }}

{% endfor %}

Write a concise answer using only the retrieved passages.
"""

# Create prompt template
prompt_builder = ChatPromptBuilder(
    template=[
        ChatMessage.from_system(system_prompt.strip()),
        ChatMessage.from_user(user_prompt.strip())
    ],
    required_variables=["question", "documents"]
)

### 6.2 Create Prompt

In [77]:
# Create prompt
prompt_result = prompt_builder.run(
    question=question,
    documents=retrieved_documents,
)

# Get prompt
messages = prompt_result["prompt"]

### 6.3 Create Response Generator

In [78]:
# Import packages
import os
from pathlib import Path
from dotenv import load_dotenv

# Get current working directory
CURRENT_DIR = Path.cwd()

# Move up to project root
ROOT_DIR = CURRENT_DIR.parents[1]

# Define path to .env
ENV_PATH = ROOT_DIR / ".env"

# Load .env content
load_dotenv(dotenv_path=ENV_PATH)

# Get API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [79]:
# Import package
from haystack.utils import Secret

# Create response generator
generator = GoogleGenAIChatGenerator(
    model="gemini-2.5-flash",
    api_key=Secret.from_token(GEMINI_API_KEY),
    generation_kwargs={"temperature": 0.2}
)

### 6.4. Generate Response

In [80]:
# Generate response
generation_result = generator.run(
    messages=messages
)

In [81]:
# Extract text response
answer = generation_result["replies"][0].text

# Print response
print(answer)

Users raised several concerns about Google's AutoLink feature:
*   It directs people to pre-selected commercial websites, which some net users found concerning [1].
*   Users were concerned that Google's dominant position could give a competitive edge to firms like Amazon [1].
*   The feature creates links based on information in a webpage without the publisher's permission, meaning websites, including online libraries or those with their own advertising, could unwillingly direct users to Amazon.com or rival services [1, 4].
*   Some users felt it would only be fair if websites had to sign up to allow the feature or receive revenue for any "click through" to a commercial site [2].
*   Users questioned if they could choose to use the service, how much Google was being paid, and if they could substitute their own companies for those chosen by Google [3].
*   There were objections to users being forced or "tricked into using the service" [3].
*   The tool was described as a "bad idea" [4]